In [1]:
def text_to_binary(text):
    return ''.join(format(ord(ch), '08b') for ch in text)

# XOR operation between two binary strings
def xor(a, b):
    result = ''
    for i in range(1, len(b)):
        result += '0' if a[i] == b[i] else '1'
    return result

# Perform Modulo-2 division
def mod2div(dividend, divisor):
    pick = len(divisor)
    tmp = dividend[0:pick]

    while pick < len(dividend):
        if tmp[0] == '1':
            tmp = xor(divisor, tmp) + dividend[pick]
        else:
            tmp = xor('0' * pick, tmp) + dividend[pick]
        pick += 1

    # Final XOR step
    if tmp[0] == '1':
        tmp = xor(divisor, tmp)
    else:
        tmp = xor('0' * pick, tmp)

    return tmp  # CRC remainder

# Sender-side: Encode data with CRC
def encodeData(data, key):
    l_key = len(key)
    appended_data = data + '0' * (l_key - 1)
    remainder = mod2div(appended_data, key)
    codeword = data + remainder
    return codeword, remainder

# Receiver-side: Check for error
def checkData(received_codeword, key):
    remainder = mod2div(received_codeword, key)
    if '1' in remainder:
        return "Error Detected"
    else:
        return "No Error"

# -------------------------------------------------
# CRC generator polynomials (binary representation)
# CRC-12   : x^12 + x^11 + x^3 + x^2 + x + 1
CRC12 = "1100000001111"

# CRC-16   : x^16 + x^15 + x^2 + 1
CRC16 = "11000000000000101"

# CRC-CCITT: x^16 + x^12 + x^5 + 1
CRCCCITT = "10001000000100001"

# Input character data
text = "HELLO"
data = text_to_binary(text)

print("Original Text :", text)
print("Binary Data   :", data)

# ---------------- CRC-12 ----------------
print("\n===== CRC-12 =====")
code12, rem12 = encodeData(data, CRC12)
print("CRC-12 Remainder :", rem12)
print("Transmitted Code :", code12)
print("Receiver Check   :", checkData(code12, CRC12))

# ---------------- CRC-16 ----------------
print("\n===== CRC-16 =====")
code16, rem16 = encodeData(data, CRC16)
print("CRC-16 Remainder :", rem16)
print("Transmitted Code :", code16)
print("Receiver Check   :", checkData(code16, CRC16))

# --------------- CRC-CCITT --------------
print("\n===== CRC-CCITT =====")
codecc, remcc = encodeData(data, CRCCCITT)
print("CRC-CCITT Remainder :", remcc)
print("Transmitted Code    :", codecc)
print("Receiver Check      :", checkData(codecc, CRCCCITT))

# Introduce an error in CRC-16 codeword
print("\n===== Error Detection Test (CRC-16) =====")
tampered = code16[:10] + ('1' if code16[10] == '0' else '0') + code16[11:]
print("Tampered Codeword :", tampered)
print("Receiver Check    :", checkData(tampered, CRC16))


Original Text : HELLO
Binary Data   : 0100100001000101010011000100110001001111

===== CRC-12 =====
CRC-12 Remainder : 001001011000
Transmitted Code : 0100100001000101010011000100110001001111001001011000
Receiver Check   : No Error

===== CRC-16 =====
CRC-16 Remainder : 0111010110001001
Transmitted Code : 01001000010001010100110001001100010011110111010110001001
Receiver Check   : No Error

===== CRC-CCITT =====
CRC-CCITT Remainder : 0101100011011010
Transmitted Code    : 01001000010001010100110001001100010011110101100011011010
Receiver Check      : No Error

===== Error Detection Test (CRC-16) =====
Tampered Codeword : 01001000011001010100110001001100010011110111010110001001
Receiver Check    : Error Detected
